In [1]:
FACTOR_ID = "K014_OFFICIAL_XGB_DUALTARGET_STUDENT_FIXEDTRAIN_NORANK"


def main(datasources, start_date, end_date):
    """Multi-resolution 4/8/16-bin path ranker; historical public factors are not model inputs."""
    import gc
    import numpy as np
    import pandas as pd
    import dai
    import xgboost as xgb
    if not hasattr(xgb, "XGBRegressor"):
        raise RuntimeError("K014 requires xgboost.XGBRegressor")
    print("K014 backend=xgboost version=" + str(getattr(xgb, "__version__", "unknown")))

    TRAIN_BAR1M = "bigalpha_2026_stock_bar1m"
    TRAIN_START = "2019-01-01 00:00:00"
    TRAIN_END = "2023-12-31 23:59:59"
    FEATURES = ['c06_1045_trade_share', 'c06_1045_amount_share', 'c06_1045_volume_share', 'v25_amount_share_dct2', 'c07_1100_trade_share', 'path_return_peak_bin', 'c05_1030_trade_share', 'c08_1115_trade_share', 'c09_1300_trade_share', 'c09_1300_amount_share', 'c09_1300_volume_share', 'v25_amount_hhi', 'v25_amount_share_dct4', 'v25_amount_entropy', 'v25_shock_peak_bin', 'c07_1100_amount_share', 'c07_1100_volume_share', 'c05_1030_amount_share', 'c05_1030_volume_share', 'c01_0930_signed_amount_share', 'state_current__v15_gap_x_prev_range', 'c04_1015_amount_share', 'c04_1015_volume_share', 'state_delta1__daily_upper_shadow', 'state_mean5__v15_gap_x_prev_range', 'c10_1315_trade_share', 'c08_1115_amount_share', 'state_std20__daily_range', 'path_return_early_minus_late', 'c08_1115_volume_share', 'path_return_slope', 'v25_return_dct1', 'v25_amount_leads_return_corr', 'state_current__v15_gap_over_prev_vol_20', 'state_mean5__daily_upper_shadow', 'c04_1015_trade_share', 'c01_0930_return', 'state_std20__v15_prev_range', 'c01_0930_volume_share', 'c01_0930_amount_share', 'state_mean5__v15_gap_over_prev_vol_20', 'state_delta1__daily_limit_pressure_balance', 'path_spread_mean_peak_bin', 'c03_1000_amount_share', 'c11_1330_trade_share', 'c03_1000_volume_share', 'state_std20__v15_trade_open5_sum_ratio20', 'c01_0930_trade_share', 'c10_1315_amount_share', 'c12_1345_trade_share', 'c10_1315_volume_share', 'c11_1330_amount_share', 'c11_1330_volume_share', 'state_std20__trade_open15_sum', 'c16_1445_amount_share', 'state_std20__daily_volume_surprise_60', 'c16_1445_volume_share', 'state_mean5__log_mid_return_open5_std', 'state_current__log_mid_return_open5_std', 'c04_1015_return', 'state_std20__trade_open5_sum', 'v25_amount_share_dct3', 'c16_1445_range', 'c02_0945_signed_amount_share', 'state_std20__daily_market_beta_60', 'c16_1445_return_std', 'state_std20__daily_close_location', 'c13_1400_trade_share', 'c15_1430_return_std', 'c03_1000_trade_share', 'c02_0945_return', 'state_std20__v15_prev_return_mean_20', 'v25_return_dct2', 'state_std20__daily_momentum_20', 'c11_1330_return_std', 'c12_1345_return_std', 'c13_1400_return_std', 'c15_1430_range', 'c16_1445_trade_share', 'state_std20__log_mid_return_open5_std', 'c01_0930_return_std', 'path_amount_share_roughness', 'c14_1415_return_std', 'state_std20__v15_open_to_prev_close_ma60', 'c14_1415_trade_share', 'c13_1400_volume_share', 'c13_1400_amount_share', 'state_current__v15_prev_range', 'state_std20__v15_gap_over_prev_vol_20', 'c13_1400_range', 'c10_1315_return_std', 'c12_1345_range', 'c14_1415_range', 'state_std20__v15_gap_x_prev_range', 'state_std20__v15_open_to_prev_close_ma20', 'v25_spread_mean_dct1', 'state_current__daily_momentum_5', 'state_current__daily_momentum_20', 'state_current__daily_momentum_60', 'state_current__daily_volatility_5', 'state_current__daily_volatility_20', 'state_current__daily_volatility_60', 'state_current__daily_market_beta_60', 'state_current__daily_amihud_20', 'state_current__daily_range', 'state_current__daily_ret_oc', 'state_current__daily_close_location', 'state_current__daily_volume_surprise_60']
    MODEL_FEATURES = ['c06_1045_trade_share', 'c06_1045_amount_share', 'c06_1045_volume_share', 'v25_amount_share_dct2', 'c07_1100_trade_share', 'path_return_peak_bin', 'c05_1030_trade_share', 'c08_1115_trade_share', 'c09_1300_trade_share', 'c09_1300_amount_share', 'c09_1300_volume_share', 'v25_amount_hhi', 'v25_amount_share_dct4', 'v25_amount_entropy', 'v25_shock_peak_bin', 'c07_1100_amount_share', 'c07_1100_volume_share', 'c05_1030_amount_share', 'c05_1030_volume_share', 'c01_0930_signed_amount_share', 'state_current__v15_gap_x_prev_range', 'c04_1015_amount_share', 'c04_1015_volume_share', 'state_delta1__daily_upper_shadow', 'state_mean5__v15_gap_x_prev_range', 'c10_1315_trade_share', 'c08_1115_amount_share', 'state_std20__daily_range', 'path_return_early_minus_late', 'c08_1115_volume_share', 'path_return_slope', 'v25_return_dct1', 'v25_amount_leads_return_corr', 'state_current__v15_gap_over_prev_vol_20', 'state_mean5__daily_upper_shadow', 'c04_1015_trade_share', 'c01_0930_return', 'state_std20__v15_prev_range', 'c01_0930_volume_share', 'c01_0930_amount_share', 'state_mean5__v15_gap_over_prev_vol_20', 'state_delta1__daily_limit_pressure_balance', 'path_spread_mean_peak_bin', 'c03_1000_amount_share', 'c11_1330_trade_share', 'c03_1000_volume_share', 'state_std20__v15_trade_open5_sum_ratio20', 'c01_0930_trade_share', 'c10_1315_amount_share', 'c12_1345_trade_share', 'c10_1315_volume_share', 'c11_1330_amount_share', 'c11_1330_volume_share', 'state_std20__trade_open15_sum', 'c16_1445_amount_share', 'state_std20__daily_volume_surprise_60', 'c16_1445_volume_share', 'state_mean5__log_mid_return_open5_std', 'state_current__log_mid_return_open5_std', 'c04_1015_return', 'state_std20__trade_open5_sum', 'v25_amount_share_dct3', 'c16_1445_range', 'c02_0945_signed_amount_share', 'state_std20__daily_market_beta_60', 'c16_1445_return_std', 'state_std20__daily_close_location', 'c13_1400_trade_share', 'c15_1430_return_std', 'c03_1000_trade_share', 'c02_0945_return', 'state_std20__v15_prev_return_mean_20', 'v25_return_dct2', 'state_std20__daily_momentum_20', 'c11_1330_return_std', 'c12_1345_return_std', 'c13_1400_return_std', 'c15_1430_range', 'c16_1445_trade_share', 'state_std20__log_mid_return_open5_std', 'c01_0930_return_std', 'path_amount_share_roughness', 'c14_1415_return_std', 'state_std20__v15_open_to_prev_close_ma60', 'c14_1415_trade_share', 'c13_1400_volume_share', 'c13_1400_amount_share', 'state_current__v15_prev_range', 'state_std20__v15_gap_over_prev_vol_20', 'c13_1400_range', 'c10_1315_return_std', 'c12_1345_range', 'c14_1415_range', 'state_std20__v15_gap_x_prev_range', 'state_std20__v15_open_to_prev_close_ma20', 'v25_spread_mean_dct1']
    BASE_STATE = ['amount_open15_sum', 'amount_open5_sum', 'daily_amihud_20', 'daily_close_location', 'daily_dist_up_10pct', 'daily_limit_pressure_balance', 'daily_lower_shadow', 'daily_market_beta_60', 'daily_momentum_20', 'daily_momentum_5', 'daily_momentum_60', 'daily_range', 'daily_resid_ret_1d', 'daily_ret_oc', 'daily_upper_shadow', 'daily_volatility_20', 'daily_volatility_5', 'daily_volatility_60', 'daily_volume_surprise_60', 'log_mid_return_open5_std', 'official_pressure3_signed_open5_mean', 'trade_open15_sum', 'trade_open5_sum', 'v15_gap_over_prev_vol_20', 'v15_gap_x_prev_range', 'v15_open_to_prev_close_ma20', 'v15_open_to_prev_close_ma5', 'v15_open_to_prev_close_ma60', 'v15_prev_intraday_return', 'v15_prev_range', 'v15_prev_return_mean_20', 'v15_prev_return_mean_5', 'v15_trade_open5_sum_ratio20', 'v15_transition_log_depth_difference', 'v15_transition_spread_difference']
    BINS = [
        ("c01_0930", 930, 944), ("c02_0945", 945, 959),
        ("c03_1000", 1000, 1014), ("c04_1015", 1015, 1029),
        ("c05_1030", 1030, 1044), ("c06_1045", 1045, 1059),
        ("c07_1100", 1100, 1114), ("c08_1115", 1115, 1129),
        ("c09_1300", 1300, 1314), ("c10_1315", 1315, 1329),
        ("c11_1330", 1330, 1344), ("c12_1345", 1345, 1359),
        ("c13_1400", 1400, 1414), ("c14_1415", 1415, 1429),
        ("c15_1430", 1430, 1444), ("c16_1445", 1445, 1456),
    ]

    def safe_div(left, right):
        return left / right.replace(0, np.nan)

    def build_daily(table, sd, ed, need_label=False):
        requested_sd, requested_ed = pd.to_datetime(sd), pd.to_datetime(ed)
        query_sd = requested_sd - pd.Timedelta(days=150)
        query_ed = requested_ed + (pd.Timedelta(days=10) if need_label else pd.Timedelta(0))
        sd, ed = query_sd, query_ed
        aggregates = []
        for name, begin, finish in BINS:
            condition = f"hhmm BETWEEN {begin} AND {finish}"
            aggregates.extend([
                f"ARG_MIN(CASE WHEN {condition} THEN mid END, CASE WHEN {condition} THEN bar_time END) AS {name}_mid_first",
                f"ARG_MAX(CASE WHEN {condition} THEN mid END, CASE WHEN {condition} THEN bar_time END) AS {name}_mid_last",
                f"STDDEV_SAMP(CASE WHEN {condition} THEN log_return END) AS {name}_return_std",
                f"SUM(CASE WHEN {condition} THEN ABS(log_return) ELSE 0.0 END) AS {name}_abs_return_sum",
                f"SUM(CASE WHEN {condition} THEN amount_step ELSE 0.0 END) AS {name}_amount_sum",
                f"SUM(CASE WHEN {condition} THEN volume_step ELSE 0.0 END) AS {name}_volume_sum",
                f"SUM(CASE WHEN {condition} THEN trade_step ELSE 0.0 END) AS {name}_trade_sum",
                f"AVG(CASE WHEN {condition} THEN spread END) AS {name}_spread_mean",
                f"AVG(CASE WHEN {condition} THEN imbalance3 END) AS {name}_imbalance3_mean",
                f"AVG(CASE WHEN {condition} THEN pressure3 END) AS {name}_pressure3_mean",
                f"AVG(CASE WHEN {condition} THEN log_depth3 END) AS {name}_log_depth3_mean",
                f"SUM(CASE WHEN {condition} THEN ofi3 ELSE 0.0 END) AS {name}_ofi3_sum",
                f"SUM(CASE WHEN {condition} THEN signed_amount ELSE 0.0 END) AS {name}_signed_amount_sum",
                f"MAX(CASE WHEN {condition} THEN high_px END) AS {name}_high",
                f"MIN(CASE WHEN {condition} THEN low_px END) AS {name}_low",
            ])
        aggregate_sql = ",\n                ".join(aggregates)
        sql = f"""
        WITH raw AS (
            SELECT
                DATE_TRUNC('day', t.date)::DATE AS trading_day,
                t.date AS bar_time,
                t.instrument::string AS instrument,
                EXTRACT(HOUR FROM t.date) * 100 + EXTRACT(MINUTE FROM t.date) AS hhmm,
                CAST(t.close AS DOUBLE) AS close_px,
                CAST(t.high AS DOUBLE) AS high_px,
                CAST(t.low AS DOUBLE) AS low_px,
                GREATEST(COALESCE(CAST(t.amount AS DOUBLE), 0.0), 0.0) AS amount_step,
                GREATEST(COALESCE(CAST(t.volume AS DOUBLE), 0.0), 0.0) AS volume_step,
                GREATEST(COALESCE(CAST(t.deal_number AS DOUBLE), 0.0), 0.0) AS trade_step,
                CAST(t.ask_price1 AS DOUBLE) AS ap1,
                CAST(t.bid_price1 AS DOUBLE) AS bp1,
                COALESCE(CAST(t.ask_volume1 AS DOUBLE), 0.0) AS av1,
                COALESCE(CAST(t.ask_volume2 AS DOUBLE), 0.0) AS av2,
                COALESCE(CAST(t.ask_volume3 AS DOUBLE), 0.0) AS av3,
                COALESCE(CAST(t.bid_volume1 AS DOUBLE), 0.0) AS bv1,
                COALESCE(CAST(t.bid_volume2 AS DOUBLE), 0.0) AS bv2,
                COALESCE(CAST(t.bid_volume3 AS DOUBLE), 0.0) AS bv3
            FROM {table} AS t
            WHERE EXTRACT(HOUR FROM t.date) * 100 + EXTRACT(MINUTE FROM t.date) BETWEEN 930 AND 1456
              AND t.ask_price1 > 0 AND t.bid_price1 > 0 AND t.close > 0
        ), geometry AS (
            SELECT *,
                (ap1 + bp1) / 2.0 AS mid,
                bv1 + bv2 + bv3 AS bid3,
                av1 + av2 + av3 AS ask3,
                bv1 + EXP(-0.3) * bv2 + EXP(-0.6) * bv3 AS weighted_bid3,
                av1 + EXP(-0.3) * av2 + EXP(-0.6) * av3 AS weighted_ask3,
                ABS(ap1 - bp1) / NULLIF((ap1 + bp1) / 2.0, 0) AS spread
            FROM raw
        ), ordered0 AS (
            SELECT *,
                LN(NULLIF(mid, 0) / NULLIF(LAG(mid) OVER (
                    PARTITION BY trading_day, instrument ORDER BY bar_time
                ), 0)) AS log_return,
                LAG(bid3) OVER (PARTITION BY trading_day, instrument ORDER BY bar_time) AS prev_bid3,
                LAG(ask3) OVER (PARTITION BY trading_day, instrument ORDER BY bar_time) AS prev_ask3
            FROM geometry
        ), state AS (
            SELECT *,
                LN(1.0 + GREATEST(bid3 + ask3, 0.0)) AS log_depth3,
                (bid3 - ask3) / NULLIF(bid3 + ask3, 0) AS imbalance3,
                ((bid3 - ask3) / NULLIF(bid3 + ask3, 0))
                    * ABS((bid3 - ask3) / NULLIF(bid3 + ask3, 0))
                    / NULLIF(SQRT(ABS(spread)) + 1e-8, 0) AS pressure3,
                ((weighted_bid3 - weighted_ask3) / NULLIF(weighted_bid3 + weighted_ask3, 0))
                    * ABS((weighted_bid3 - weighted_ask3) / NULLIF(weighted_bid3 + weighted_ask3, 0))
                    / NULLIF(SQRT(ABS(spread)) + 1e-8, 0) AS official_pressure3_signed,
                (bid3 - prev_bid3) - (ask3 - prev_ask3) AS ofi3,
                amount_step * SIGN(COALESCE(log_return, 0.0)) AS signed_amount
            FROM ordered0
        )
        SELECT
            trading_day AS date,
            instrument,
            ARG_MIN(close_px, bar_time) AS open,
            MAX(high_px) AS high,
            MIN(low_px) AS low,
            ARG_MAX(close_px, bar_time) AS close,
            AVG(spread) AS spread_full_mean,
            AVG(CASE WHEN hhmm BETWEEN 930 AND 934 THEN spread END) AS spread_open5_mean,
            STDDEV_SAMP(CASE WHEN hhmm BETWEEN 930 AND 934 THEN log_return END) AS log_mid_return_open5_std,
            AVG(log_depth3) AS log_depth3_full_mean,
            AVG(CASE WHEN hhmm BETWEEN 930 AND 934 THEN log_depth3 END) AS log_depth3_open5_mean,
            AVG(CASE WHEN hhmm BETWEEN 930 AND 934 THEN official_pressure3_signed END) AS official_pressure3_signed_open5_mean,
            SUM(CASE WHEN hhmm BETWEEN 930 AND 934 THEN amount_step ELSE 0.0 END) AS amount_open5_sum,
            SUM(CASE WHEN hhmm BETWEEN 930 AND 934 THEN trade_step ELSE 0.0 END) AS trade_open5_sum,
            SUM(amount_step) AS amount_total,
            SUM(volume_step) AS volume_total,
            SUM(trade_step) AS trade_total,
            SUM(ABS(log_return)) AS abs_return_total,
            {aggregate_sql}
        FROM state
        GROUP BY trading_day, instrument
        """
        frame = dai.query(sql, filters={"date": [sd, ed]}, compression=True).df()
        if frame.empty:
            return frame
        frame["date"] = pd.to_datetime(frame["date"])
        frame["instrument"] = frame["instrument"].astype(str)
        numeric = [column for column in frame.columns if column not in ["date", "instrument"]]
        frame[numeric] = frame[numeric].apply(pd.to_numeric, errors="coerce")
        for name, _, _ in BINS:
            frame[f"{name}_return"] = np.log(safe_div(frame[f"{name}_mid_last"], frame[f"{name}_mid_first"]))
            frame[f"{name}_range"] = safe_div(frame[f"{name}_high"] - frame[f"{name}_low"], frame[f"{name}_mid_last"])
            frame[f"{name}_amount_share"] = safe_div(frame[f"{name}_amount_sum"], frame["amount_total"])
            frame[f"{name}_volume_share"] = safe_div(frame[f"{name}_volume_sum"], frame["volume_total"])
            frame[f"{name}_trade_share"] = safe_div(frame[f"{name}_trade_sum"], frame["trade_total"])
            frame[f"{name}_rv_share"] = safe_div(frame[f"{name}_abs_return_sum"], frame["abs_return_total"])
            frame[f"{name}_ofi3_norm"] = safe_div(frame[f"{name}_ofi3_sum"], np.exp(frame[f"{name}_log_depth3_mean"]) + 1.0)
            frame[f"{name}_signed_amount_share"] = safe_div(frame[f"{name}_signed_amount_sum"], frame["amount_total"])
        names = [name for name, _, _ in BINS]
        weights = np.arange(16, dtype="float32") - 7.5
        for stem in ("return", "amount_share", "rv_share", "spread_mean", "imbalance3_mean", "pressure3_mean", "ofi3_norm"):
            columns = [f"{name}_{stem}" for name in names]
            frame[f"path_{stem}_early_minus_late"] = frame[columns[:8]].mean(axis=1) - frame[columns[8:]].mean(axis=1)
            frame[f"path_{stem}_slope"] = frame[columns].mul(weights, axis=1).sum(axis=1)
            frame[f"path_{stem}_roughness"] = frame[columns].diff(axis=1).abs().sum(axis=1)
            frame[f"path_{stem}_peak_bin"] = frame[columns].fillna(-np.inf).to_numpy().argmax(axis=1).astype("float32") / 15.0
        state_path = pd.concat([
            (frame[f"{name}_return"] > 0).astype("int8")
            + 2 * (frame[f"{name}_imbalance3_mean"] > 0).astype("int8")
            for name in names
        ], axis=1)
        frame["path_state_transitions"] = state_path.diff(axis=1).iloc[:, 1:].ne(0).sum(axis=1).astype("float32") / 15.0
        return_signs = pd.concat([np.sign(frame[f"{name}_return"]) for name in names], axis=1)
        frame["path_return_sign_flips"] = return_signs.diff(axis=1).iloc[:, 1:].ne(0).sum(axis=1).astype("float32") / 15.0

        def aggregate(columns, stem):
            values = frame[columns]
            if stem == "return_std":
                return np.sqrt(values.pow(2).sum(axis=1))
            if stem == "range":
                return values.max(axis=1)
            if stem in {"return", "amount_share", "volume_share", "trade_share", "rv_share", "ofi3_norm", "signed_amount_share"}:
                return values.sum(axis=1)
            return values.mean(axis=1)

        core_stems = (
            "return", "return_std", "range", "amount_share", "volume_share", "trade_share", "rv_share",
            "spread_mean", "imbalance3_mean", "pressure3_mean", "log_depth3_mean", "ofi3_norm", "signed_amount_share",
        )
        additions = {}
        for width, prefix in ((2, "m8"), (4, "m4")):
            for group_index, start in enumerate(range(0, 16, width), 1):
                group = names[start : start + width]
                for stem in core_stems:
                    additions[f"{prefix}_{group_index:02d}_{stem}"] = aggregate([f"{name}_{stem}" for name in group], stem).astype("float32")
        positions = np.arange(16, dtype="float64") + 0.5
        for stem in ("return", "amount_share", "rv_share", "spread_mean", "imbalance3_mean", "pressure3_mean", "ofi3_norm"):
            columns = [f"{name}_{stem}" for name in names]
            values = frame[columns].to_numpy(dtype="float64", copy=True)
            row_fill = pd.DataFrame(values).median(axis=1).fillna(0.0).to_numpy()
            missing = ~np.isfinite(values)
            values[missing] = np.take(row_fill, np.where(missing)[0])
            for coefficient in range(1, 5):
                basis = np.cos(np.pi * coefficient * positions / 16.0)
                additions[f"v25_{stem}_dct{coefficient}"] = (values @ basis / 16.0).astype("float32")
            left, right = values[:, :-1], values[:, 1:]
            left = left - left.mean(axis=1, keepdims=True)
            right = right - right.mean(axis=1, keepdims=True)
            denominator = np.sqrt(np.sum(left * left, axis=1) * np.sum(right * right, axis=1))
            additions[f"v25_{stem}_autocorr1"] = np.divide(
                np.sum(left * right, axis=1), denominator,
                out=np.zeros(len(values), dtype="float64"), where=denominator > 1e-12,
            ).astype("float32")
        returns = frame[[f"{name}_return" for name in names]].fillna(0.0).to_numpy(dtype="float64")
        amount = frame[[f"{name}_amount_share" for name in names]].fillna(0.0).clip(lower=0).to_numpy(dtype="float64")
        cumulative = np.cumsum(returns, axis=1)
        additions["v25_path_max_drawdown"] = np.max(np.maximum.accumulate(cumulative, axis=1) - cumulative, axis=1).astype("float32")
        additions["v25_path_max_runup"] = np.max(cumulative - np.minimum.accumulate(cumulative, axis=1), axis=1).astype("float32")
        peak = np.argmax(np.abs(returns), axis=1)
        shock = returns[np.arange(len(frame)), peak]
        after = np.arange(16)[None, :] > peak[:, None]
        additions["v25_shock_peak_bin"] = (peak / 15.0).astype("float32")
        additions["v25_shock_recovery"] = (-np.sign(shock) * np.sum(np.where(after, returns, 0.0), axis=1)).astype("float32")
        normalized_amount = amount / np.where(amount.sum(axis=1, keepdims=True) > 0, amount.sum(axis=1, keepdims=True), 1.0)
        additions["v25_amount_entropy"] = (-np.sum(normalized_amount * np.log(normalized_amount + 1e-12), axis=1) / np.log(16.0)).astype("float32")
        additions["v25_amount_hhi"] = np.sum(normalized_amount * normalized_amount, axis=1).astype("float32")
        def row_corr(left, right):
            left = left - left.mean(axis=1, keepdims=True)
            right = right - right.mean(axis=1, keepdims=True)
            denominator = np.sqrt(np.sum(left * left, axis=1) * np.sum(right * right, axis=1))
            return np.divide(np.sum(left * right, axis=1), denominator, out=np.zeros(len(left)), where=denominator > 1e-12)
        additions["v25_amount_leads_return_corr"] = row_corr(amount[:, :-1], returns[:, 1:]).astype("float32")
        additions["v25_return_leads_amount_corr"] = row_corr(returns[:, :-1], amount[:, 1:]).astype("float32")
        frame = pd.concat([frame, pd.DataFrame(additions, index=frame.index)], axis=1, copy=False)

        # Daily/cross-day state; all rolling inputs are current or older.
        universe = stock_pool(sd, ed)
        frame = pd.merge(frame, universe, on=["date", "instrument"], how="inner", validate="one_to_one")
        frame = frame.sort_values(["instrument", "date"], kind="mergesort").reset_index(drop=True)
        grouped = frame.groupby("instrument", sort=False)
        frame["_history_ready"] = grouped.cumcount().ge(19)
        previous_close = grouped["close"].shift(1)
        previous_open = grouped["open"].shift(1)
        price_range = (frame["high"] - frame["low"]).abs()
        frame["daily_ret_1d"] = safe_div(frame["close"], previous_close) - 1.0
        frame["daily_ret_oc"] = safe_div(frame["close"], frame["open"]) - 1.0
        frame["daily_range"] = safe_div(price_range, frame["close"])
        frame["daily_close_location"] = safe_div(frame["close"] - frame["low"], price_range) - 0.5
        frame["daily_upper_shadow"] = safe_div(frame["high"] - np.maximum(frame["open"], frame["close"]), price_range)
        frame["daily_lower_shadow"] = safe_div(np.minimum(frame["open"], frame["close"]) - frame["low"], price_range)
        for window in (5, 20, 60):
            frame[f"daily_momentum_{window}"] = safe_div(frame["close"], grouped["close"].shift(window)) - 1.0
            frame[f"daily_volatility_{window}"] = grouped["daily_ret_1d"].transform(
                lambda values: values.rolling(window, min_periods=max(3, window // 2)).std(ddof=1)
            )
        adv_amount20 = grouped["amount_total"].transform(lambda values: values.rolling(20, min_periods=10).mean())
        adv_volume60 = grouped["volume_total"].transform(lambda values: values.rolling(60, min_periods=30).mean())
        frame["daily_amihud_20"] = safe_div(frame["daily_ret_1d"].abs(), adv_amount20)
        frame["daily_volume_surprise_60"] = safe_div(frame["volume_total"], adv_volume60)
        market = frame.groupby("date", sort=False)["daily_ret_1d"].median().rename("market_ret_median")
        frame = frame.merge(market.reset_index(), on="date", how="left", validate="many_to_one")
        frame = frame.sort_values(["instrument", "date"], kind="mergesort").reset_index(drop=True)
        grouped = frame.groupby("instrument", sort=False)
        previous_close = grouped["close"].shift(1)
        previous_open = grouped["open"].shift(1)
        def rolling_beta(group):
            covariance = group["daily_ret_1d"].rolling(60, min_periods=30).cov(group["market_ret_median"])
            variance = group["market_ret_median"].rolling(60, min_periods=30).var(ddof=1)
            return covariance / variance.where(variance.abs() > 1e-12)
        frame["daily_market_beta_60"] = frame.groupby("instrument", group_keys=False, sort=False)[
            ["daily_ret_1d", "market_ret_median"]
        ].apply(rolling_beta)
        frame["daily_resid_ret_1d"] = frame["daily_ret_1d"] - frame["daily_market_beta_60"] * frame["market_ret_median"]
        frame["daily_dist_up_10pct"] = safe_div(previous_close * 1.10 - frame["close"], previous_close)
        daily_dist_down = safe_div(frame["close"] - previous_close * 0.90, previous_close)
        frame["daily_limit_pressure_balance"] = (
            (frame["daily_dist_up_10pct"] <= 0.01).astype("float32")
            - (daily_dist_down <= 0.01).astype("float32")
        )
        overnight_gap = safe_div(frame["open"], previous_close) - 1.0
        frame["v15_prev_intraday_return"] = safe_div(previous_close, previous_open) - 1.0
        frame["v15_prev_range"] = safe_div(grouped["high"].shift(1) - grouped["low"].shift(1), previous_close)
        frame["v15_gap_x_prev_range"] = overnight_gap * frame["v15_prev_range"]
        for window in (5, 20, 60):
            prior_mean = grouped["close"].transform(
                lambda values, w=window: values.shift(1).rolling(w, min_periods=max(5, w // 4)).mean()
            )
            frame[f"v15_open_to_prev_close_ma{window}"] = safe_div(frame["open"], prior_mean) - 1.0
        shifted_return = grouped["daily_ret_1d"].shift(1)
        for window in (5, 20):
            prior_mean = shifted_return.groupby(frame["instrument"], sort=False).transform(
                lambda values, w=window: values.rolling(w, min_periods=max(3, w // 4)).mean()
            )
            prior_vol = shifted_return.groupby(frame["instrument"], sort=False).transform(
                lambda values, w=window: values.rolling(w, min_periods=max(3, w // 4)).std(ddof=1)
            )
            frame[f"v15_prev_return_mean_{window}"] = prior_mean
            if window == 20:
                frame["v15_gap_over_prev_vol_20"] = safe_div(overnight_gap, prior_vol.abs() + 1e-6)
        previous_trade_open5_mean20 = grouped["trade_open5_sum"].transform(
            lambda values: values.shift(1).rolling(20, min_periods=5).mean()
        )
        frame["v15_trade_open5_sum_ratio20"] = safe_div(frame["trade_open5_sum"], previous_trade_open5_mean20) - 1.0
        frame["v15_transition_spread_difference"] = frame["spread_open5_mean"] - grouped["spread_full_mean"].shift(1)
        frame["v15_transition_log_depth_difference"] = frame["log_depth3_open5_mean"] - grouped["log_depth3_full_mean"].shift(1)
        frame["amount_open5_sum"] = frame["amount_open5_sum"]
        frame["trade_open5_sum"] = frame["trade_open5_sum"]
        frame["amount_open15_sum"] = frame["c01_0930_amount_sum"]
        frame["trade_open15_sum"] = frame["c01_0930_trade_sum"]

        state_rank = frame.groupby("date", sort=False)[BASE_STATE].rank(pct=True, method="average").sub(0.5)
        state_rank = state_rank.replace([np.inf, -np.inf], np.nan).fillna(0.0).astype("float32")
        state_rank["instrument"] = frame["instrument"].to_numpy()
        for column in BASE_STATE:
            values = state_rank[column]
            state_group = values.groupby(state_rank["instrument"], sort=False)
            frame[f"state_current__{column}"] = values
            frame[f"state_mean5__{column}"] = state_group.transform(
                lambda series: series.rolling(5, min_periods=3).mean()
            )
            frame[f"state_std20__{column}"] = state_group.transform(
                lambda series: series.rolling(20, min_periods=10).std(ddof=0)
            )
            frame[f"state_delta1__{column}"] = values - state_group.shift(1)

        intraday_features = [column for column in FEATURES if not column.startswith("state_")]
        ranked_intraday = frame.groupby("date", sort=False)[intraday_features].rank(pct=True, method="average").sub(0.5)
        frame[intraday_features] = ranked_intraday.replace([np.inf, -np.inf], np.nan).fillna(0.0).astype("float32")
        frame = frame.replace([np.inf, -np.inf], np.nan)
        if need_label:
            frame = frame.sort_values(["instrument", "date"], kind="mergesort")
            frame["vwap"] = safe_div(frame["amount_total"], frame["volume_total"])
            grouped = frame.groupby("instrument", sort=False)
            for column in ("open", "close", "amount_total", "volume_total", "vwap"):
                frame[f"next_{column}"] = grouped[column].shift(-1)
            open_gap = safe_div(frame["next_open"], frame["close"]) - 1.0
            has_price = frame[["next_open", "next_close", "next_vwap"]].notna().all(axis=1)
            suspended = (~has_price) | (frame["next_amount_total"] <= 0) | (frame["next_volume_total"] <= 0)
            next_amount_rank = frame.groupby("date", sort=False)["next_amount_total"].rank(pct=True)
            tradable = (
                (~suspended)
                & (~(open_gap.abs() >= 0.095).fillna(False))
                & (~(open_gap.abs() > 0.30).fillna(True))
                & (next_amount_rank >= 0.05).fillna(False)
            )
            p0_return = safe_div(frame["next_close"], frame["next_vwap"]) - 1.0
            frame["label"] = p0_return.where(tradable)
            frame["raw_label"] = safe_div(frame["next_close"], frame["close"]) - 1.0
        keep = ["date", "instrument", "close", "_history_ready", *FEATURES] + (["label", "raw_label"] if need_label else [])
        frame = frame[(frame["date"] >= requested_sd) & (frame["date"] <= requested_ed)]
        return frame[keep].reset_index(drop=True)

    def cross_sectional_features(frame):
        keys = ["date", "instrument"] + (["label", "raw_label"] if "label" in frame.columns else [])
        result = frame[keys + FEATURES].copy()
        result[FEATURES] = result[FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0.0).astype("float32")
        return result

    def stock_pool(sd, ed):
        pool = dai.query(
            "SELECT date, instrument FROM bigalpha_2026_instruments",
            filters={"date": [sd, ed]}, compression=True,
        ).df()
        pool["date"] = pd.to_datetime(pool["date"])
        pool["instrument"] = pool["instrument"].astype(str)
        return pool

    train_parts = []
    for year in range(2019, 2024):
        sd, ed = f"{year}-01-01 00:00:00", f"{year}-12-31 23:59:59"
        chunk = build_daily(TRAIN_BAR1M, sd, ed, need_label=True)
        chunk = pd.merge(chunk, stock_pool(sd, ed), on=["date", "instrument"], how="inner")
        chunk = chunk.dropna(subset=["label", "raw_label"])
        chunk = chunk[chunk["_history_ready"]].copy()
        chunk = cross_sectional_features(chunk)
        chunk["sample_hash"] = pd.util.hash_pandas_object(
            chunk[["date", "instrument"]], index=False
        ).astype("uint64")
        chunk = chunk.sort_values(["date", "sample_hash"], kind="mergesort").groupby(
            "date", sort=False
        ).head(320).drop(columns="sample_hash")
        train_parts.append(chunk)
        del chunk
        gc.collect()
    train = pd.concat(train_parts, ignore_index=True, copy=False).sort_values(["date", "instrument"], kind="mergesort")
    del train_parts
    causal_columns = ['state_current__daily_momentum_5', 'state_current__daily_momentum_20', 'state_current__daily_momentum_60', 'state_current__daily_volatility_5', 'state_current__daily_volatility_20', 'state_current__daily_volatility_60', 'state_current__daily_market_beta_60', 'state_current__daily_amihud_20', 'state_current__daily_range', 'state_current__daily_ret_oc', 'state_current__daily_close_location', 'state_current__daily_volume_surprise_60']

    def orthogonal_target(label_column):
        ranked = train.groupby("date", sort=False)[label_column].rank(pct=True, method="average").sub(0.5)
        target = ranked.to_numpy(dtype="float64", copy=True)
        style_values = train[causal_columns].to_numpy(dtype="float64", copy=False)
        group_id = pd.factorize(train["date"], sort=False)[0].astype("int32")
        result = np.zeros(len(train), dtype="float32")
        for group in range(int(group_id.max()) + 1):
            rows = np.flatnonzero(group_id == group)
            y_day = target[rows]
            x_day = style_values[rows]
            x_design = np.column_stack([np.ones(len(rows), dtype="float64"), x_day])
            gram = x_design.T @ x_design
            penalty = np.eye(gram.shape[0], dtype="float64") * 1e-3
            penalty[0, 0] = 0.0
            beta = np.linalg.solve(gram + penalty, x_design.T @ y_day)
            residual = y_day - x_design @ beta
            result[rows] = pd.Series(residual).rank(pct=True, method="average").sub(0.5).to_numpy(dtype="float32")
        return result

    p0_target = orthogonal_target("label")
    raw_target = orthogonal_target("raw_label")
    dual_target = (0.60 * p0_target + 0.40 * raw_target).astype("float32")
    x_train = train[MODEL_FEATURES].to_numpy(dtype="float32", copy=False)
    model = xgb.XGBRegressor(
        objective="reg:squarederror", n_estimators=620, max_depth=4,
        learning_rate=0.025, min_child_weight=90, subsample=0.82,
        colsample_bytree=0.72, reg_alpha=3.0, reg_lambda=60.0,
        tree_method="hist", n_jobs=4, random_state=2026074014,
    )
    model.fit(x_train, dual_target, verbose=False)
    del train, p0_target, raw_target, dual_target, x_train
    gc.collect()

    requested_start, requested_end = pd.to_datetime(start_date), pd.to_datetime(end_date)
    results = []
    for year in range(requested_start.year, requested_end.year + 1):
        sd = max(requested_start, pd.Timestamp(year=year, month=1, day=1))
        ed = min(requested_end, pd.Timestamp(year=year, month=12, day=31, hour=23, minute=59, second=59))
        pool = stock_pool(sd, ed)
        test = build_daily(datasources["bar1m"], sd, ed, need_label=False)
        if test.empty:
            pool["factor"] = 0.0
            results.append(pool[["date", "instrument", "factor"]])
            continue
        test = cross_sectional_features(test)
        x_test = test[MODEL_FEATURES].to_numpy(dtype="float32", copy=False)
        test["factor"] = model.predict(x_test)
        result = pd.merge(pool, test[["date", "instrument", "factor"]], on=["date", "instrument"], how="left")
        result["factor"] = pd.to_numeric(result["factor"], errors="coerce").replace([np.inf, -np.inf], np.nan)
        result["factor"] = result.groupby("date", sort=False)["factor"].transform(lambda values: values.fillna(values.median())).fillna(0.0)
        results.append(result[["date", "instrument", "factor"]])
        del pool, test, result
        gc.collect()
    result = pd.concat(results, ignore_index=True, copy=False)
    result = result[(result["date"] >= requested_start) & (result["date"] <= requested_end)]
    result["factor"] = pd.to_numeric(result["factor"], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return result[["date", "instrument", "factor"]].reset_index(drop=True)


if __name__ == "__main__":
    from bigmodule import M
    import dai
    datasources = {"bar1m": "bigalpha_2026_stock_bar1m"}
    start_date, end_date = "2024-01-01 00:00:00", "2024-12-31 23:59:59"
    factor_data = main(datasources, start_date, end_date)
    factor_pool = dai.query("SELECT * FROM bigalpha_2026_factorlib", filters={"date": [start_date, end_date]}).df()
    result = M.bigalpha_eval._latest(
        factor_data=factor_data, factor_pool=factor_pool, start_date=start_date,
        end_date=end_date, process_pools=False, show=True,
    )


K001 backend=xgboost version=1.7.3
